# 07 — Análisis Exploratorio
## Consultas sobre la Capa Gold

Notebook de exploración ad-hoc sobre los datos de Gold.
Lee directamente los Parquet del modelo dimensional y calcula KPIs.

**Requiere:** Haber ejecutado los notebooks 04, 05 y 06 (Gold)


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pyspark.sql import functions as F
from pyspark.sql.window import Window

from src.paths import GOLD, str_path
from src.spark_utils import get_spark

spark = get_spark(app_name="MEF_Analisis", memory="4g")
print("Listo para análisis")

## 1. Carga del Modelo Dimensional

In [ ]:
def load_gold(key):
    path = str_path(GOLD[key])
    df = spark.read.parquet(path)
    n = df.count()
    print(f"  {key:<35} {n:>10,} filas | {len(df.columns)} cols")
    return df

print("Cargando Gold SIAF:")
fact    = load_gold("fact_ejecucion_presupuestal")
d_tiempo = load_gold("dim_tiempo")
d_geo    = load_gold("dim_geografia")
d_muni   = load_gold("dim_municipalidad")
d_clas   = load_gold("dim_clasificacion_ingreso")
d_fin    = load_gold("dim_financiamiento")

## 2. KPI: Ejecución Presupuestal por Año

In [ ]:
print("Ejecución Presupuestal Total por Año (en millones de S/.):\n")
(
    fact.join(d_tiempo, "SK_Tiempo")
    .groupBy("Ano")
    .agg(
        F.sum("Monto_PIM").alias("PIM_Total"),
        F.sum("Monto_Recaudado").alias("Recaudado_Total"),
        F.count("*").alias("N_Registros")
    )
    .withColumn("Tasa_Avance_%", F.round(F.col("Recaudado_Total") / F.col("PIM_Total") * 100, 2))
    .withColumn("PIM_MM", F.round(F.col("PIM_Total") / 1_000_000, 1))
    .withColumn("Recaudado_MM", F.round(F.col("Recaudado_Total") / 1_000_000, 1))
    .select("Ano", "PIM_MM", "Recaudado_MM", "Tasa_Avance_%", "N_Registros")
    .orderBy("Ano")
    .show()
)

## 3. KPI: Top 10 Departamentos por Tasa de Avance

In [ ]:
print("Top 10 Departamentos por Tasa de Avance (último año disponible):\n")

ultimo_ano = fact.join(d_tiempo, "SK_Tiempo").agg(F.max("Ano")).collect()[0][0]
print(f"  Año de análisis: {ultimo_ano}\n")

(
    fact.join(d_tiempo, "SK_Tiempo").filter(F.col("Ano") == ultimo_ano)
    .join(d_geo, "SK_Geografia")
    .groupBy("Nombre_Departamento")
    .agg(
        F.sum("Monto_PIM").alias("PIM"),
        F.sum("Monto_Recaudado").alias("Recaudado"),
        F.countDistinct("SK_Municipalidad").alias("N_Municipalidades")
    )
    .withColumn("Tasa_%", F.round(F.col("Recaudado") / F.col("PIM") * 100, 2))
    .orderBy(F.col("Tasa_%").desc())
    .show(10, truncate=False)
)

## 4. KPI: Distribución de Calidad de Datos

In [ ]:
print("Distribución SK_Calidad:")
calidad_map = {
    1: "Limpio",
    2: "Advertencias menores",
    3: "Inconsistencias de catálogo",
    4: "Error crítico"
}
total = fact.count()
(
    fact.groupBy("SK_Calidad")
    .count()
    .withColumn("Pct", F.round(F.col("count") / total * 100, 2))
    .orderBy("SK_Calidad")
    .show()
)
print(f"  Total registros Fact: {total:,}")

## 5. KPI: Top Fuentes de Financiamiento

In [ ]:
print("PIM por Fuente de Financiamiento (todos los años):\n")
(
    fact.join(d_fin, "SK_Financiamiento")
    .groupBy("Fuente_Financiamiento")
    .agg(
        F.round(F.sum("Monto_PIM") / 1_000_000, 1).alias("PIM_MM"),
        F.round(F.sum("Monto_Recaudado") / 1_000_000, 1).alias("Recaudado_MM")
    )
    .orderBy(F.col("PIM_MM").desc())
    .show(10, truncate=False)
)

## 6. SISMEPRE — Vista Rápida

In [ ]:
from src.paths import GOLD

sismepre_keys = [
    "sismepre_formulario", "sismepre_preguntas", "sismepre_respuestas",
    "sismepre_estadistica", "sismepre_esat_atm",
    "sismepre_entidad_estado", "sismepre_ano_aplicacion"
]

print("SISMEPRE Gold Summary:\n")
for key in sismepre_keys:
    p = GOLD[key]
    if p.exists() and any(p.rglob("*.parquet")):
        df_s = spark.read.parquet(str_path(p))
        print(f"  {key:<35} {df_s.count():>8,} filas | {len(df_s.columns)} cols")
    else:
        print(f"  {key:<35} sin datos")